# 2pt 1-State Fit Template

This notebook is a runnable example of the 1-state two-point fit workflow using repository-relative paths. Edit the single `workflow_config` block below, run the validation cell, and then execute the workflow cell to write the fit outputs under `results_dir`.


## Imports / Setup


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import matplotlib
matplotlib.rcParams.update({
    "font.family": "Times New Roman",
    "mathtext.fontset": "custom",
    "mathtext.rm": "Times New Roman",
    "mathtext.it": "Times New Roman:italic",
    "mathtext.bf": "Times New Roman:bold",
})

from lqcd_analysis.notebook_workflows import (
    pretty_print_config,
    render_nstate_fit_input_text,
    run_nstate_fit_from_notebook,
    validate_nstate_notebook_config,
)


## User Inputs

The fields below mirror the current plain-text N-state fit input format. This notebook is configured for 1-state fits only. The defaults point to realistic example data and output into `examples/outputs/` under the notebook root.


In [ ]:
workflow_config = {'title_pattern': 'l64c64a076_m140_fit_k0_pz*',
 'ns': 64,
 'nt': 64,
 'lattice_spacing_fm': 0.076,

 # Data settings
 'c2pt': '/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/l64c64a076_m140/data/c2pt_csv/c2pt_5_5_k0_pz*_real.csv',
 'pzlist': [0],
 'fold_t': 'none',
 'model': 'normal',
 'fit_mode': 'uncorrelated',

 # Fit settings
 'nstates': 1,
 'tmin_window': {0: [0, 4]},
 'tmax': {0: 12},
 'binsize': 1,
 'bootstrap_samples': 64,
 'bootstrap_size': 64,
 'seed': 2026,

 # Prior settings
 'pz0_ground_energy': 0.42,
 'fix_ground_energy_from_dispersion': True,
 # 1-state has no low-state prior
 'plot': True,
 'results_dir': '/Users/xiang/Desktop/codes/lat-hadron-analysis-DA/examples/outputs/nstate_fit_1state_notebook'}
workflow_config


## Option Guide

Edit only `workflow_config` in the cell above for normal usage.
The keys are grouped by comments so data settings, fit settings, and output settings stay easy to scan.

### Data settings
- `title_pattern`: Output title pattern. Use `*` where the momentum index `pz` should be inserted.
- `ns`: Spatial lattice extent `Ns`.
- `nt`: Temporal lattice extent `Nt`.
- `lattice_spacing_fm`: Lattice spacing in fm for metadata and summaries.
- `c2pt`: Correlator CSV path or wildcard pattern. Keep `*` in the filename when using multiple `pz` values.
- `pzlist`: List of momentum indices to analyze, for example `[0]` or `[0, 1]`.
- `fold_t`: Time-folding mode before fitting.
  Choices: `"none"` or `False` = no folding; `"periodic"` or `True` = symmetric fold; `"antiperiodic"` = antisymmetric fold.
- `model`: Correlator model.
- `fit_mode`: Statistical error model used in the nonlinear fit.
  Choices: `"uncorrelated"` = diagonal fit with per-time-slice bootstrap standard deviations; `"correlated"` = full covariance fit using one shared covariance matrix built from the full bootstrap ensemble and reused for the mean fit and bootstrap fits. If the correlated fit fails for a sample or the window covariance cannot be factorized, the code falls back to a diagonal fit built from the covariance diagonal.

### Fit settings
- `nstates`: Number of states to fit. This template is locked to `1`.
- `tmin_window`: Preferred default notebook-facing fit-window form. Use a dictionary like `{0: [4, 12], 5: [6, 12]}` to set one `[tmin, tmax]` window per momentum. The notebook helper materializes this into the backend fit-window table format automatically.
- `binsize`: Integer configuration bin size. Use `1` for no binning.
- `bootstrap_samples`: Number of bootstrap resamples. `None` lets the backend choose automatically.
- `bootstrap_size`: Number of binned configurations drawn per bootstrap sample. `None` uses the backend default.
- `seed`: Random seed for reproducible bootstrap sampling.

### Prior settings
- `pz0_ground_energy`: Optional pz=0 ground-state energy in lattice units. When provided, it serves as the dispersion-reference input used by `fix_ground_energy_from_dispersion`.
- `fix_ground_energy_from_dispersion`: Optional boolean. When `true`, the nonlinear fit fixes the ground-state energy to that same dispersion target. This is the closest repository-native analogue to the legacy fixed-`E0` setup and is the recommended default when you trust the dispersion anchor.
- `plot`: Whether to generate plots automatically.
  Choices: `True` or `False`.
- `results_dir`: Output directory. If omitted or set to `None`, outputs go to the notebook working directory.

Practical note:
- When `tmin_window` is provided for a momentum, the fit scans `tmin` from 0 up to the window end while keeping that momentum's `tmax` fixed.
- The warm-start initial guess comes from the lower-state fit output for the same state count.
- The soft priors are disabled in this template because it is 1-state only.
- Fit tables include `fallback_uncorrelated_successes`, the number of bootstrap samples in a given `tmin` window that succeeded only after falling back from the correlated fit to a diagonal fit.


## Input Summary / Validation

This notebook follows the same single-config pattern as the TGEVP template.
Fields that belong to the plain-text input file are rendered below; notebook-only runtime fields such as `results_dir` stay in the same config for convenience.


In [ ]:
print(pretty_print_config(workflow_config))
print(render_nstate_fit_input_text(workflow_config))
parsed_nstate = validate_nstate_notebook_config(workflow_config)
parsed_nstate


## Run Analysis


In [ ]:
nstate_outputs = run_nstate_fit_from_notebook(workflow_config)
for path in nstate_outputs:
    print(path)


## Inspect Outputs

The fit writes tables, bootstrap samples, plots, and a plotting notebook under `examples/outputs/`.


In [ ]:
for path in nstate_outputs:
    print(Path(path).name)
